# Week 04 — Baseline Action Score
**Lane:** CTR / Engagement Opportunity Scoring (Lane 4)
**Builds on:** `work/research_question_framing.md`, `work/ml_task_framing.md`, `notebooks/w03_data_contract.ipynb`
**Dataset:** starter slice, `data/raw/content_refresh_anonymized.csv` (30,000 rows x 44 columns)
**Seed:** 42 (fixed below, for reproducibility)

This notebook does three things:
1. Checks two signals my rule leans on, each with a printed bucket table (`n` included) and a one-word verdict.
2. Encodes ONE rule -- one score, one reason code per row, one action label -- and writes the ranked queue to `work/outputs/baseline_action_score.csv`.
3. Reviews my own top 10 with a skeptic's eye: what got flagged, why, and what would prove it wrong.

No future-window inputs (`impressions_last_30d`/`impressions_prev_30d` and siblings, `trend_pct`) and no label-derived inputs (`trend_direction`) are used anywhere below -- confirmed in the self-check at the end.

Frame everything here as observed / measured / directional / decision-support, per `DATA_USE.md` -- not a causal claim about what will happen if a page is edited.

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rida-create/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # walk up from wherever this notebook is running until the repo root is found
    cwd = Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            os.chdir(candidate)
            break

import json
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)

SEED = 42
np.random.seed(SEED)

RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")
OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print("Loaded:", df.shape, "from", RAW_PATH.resolve())

Loaded: (30000, 44) from /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [2]:
base = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
print(f"Base rows: {len(base):,} / {len(df):,}")

Base rows: 30,000 / 30,000


In [3]:
sig1 = base[base["avg_position"] > 0].copy()  # 0 means "no position data", not position zero

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
bucket_1_raw = (
    sig1.groupby("position_tier")["ctr"]
    .agg(n="count", mean_ctr="mean", median_ctr="median")
    .reindex(tier_order)
)
print("Signal 1 -- CTR by position tier (no volume floor):")
print(bucket_1_raw)

Signal 1 -- CTR by position tier (no volume floor):
                   n  mean_ctr  median_ctr
position_tier                             
top_3           1116  2.764453        0.00
page_1         11814  0.652467        0.16
striking        7304  0.323239        0.11
page_3_5        7242  0.222484        0.03
deep            1319  0.150212        0.00


In [4]:
zero_and_volume = base.loc[sig1.index].groupby("position_tier").agg(
    n=("ctr", "count"),
    pct_zero_ctr=("ctr", lambda s: (s == 0).mean() * 100),
    median_impressions=("impressions_90d", "median"),
).reindex(tier_order)
print(zero_and_volume)

                   n  pct_zero_ctr  median_impressions
position_tier                                         
top_3           1116     52.329749                53.0
page_1         11814     34.179787              1179.5
striking        7304     39.156627               874.5
page_3_5        7242     47.293565               811.5
deep            1319     83.927218               218.0


In [5]:
sig1_floor = sig1[sig1["impressions_90d"] >= 300]
bucket_1_floor = (
    sig1_floor.groupby("position_tier")["ctr"]
    .agg(n="count", mean_ctr="mean", median_ctr="median")
    .reindex(tier_order)
)
print("Signal 1 -- CTR by position tier (impressions_90d >= 300):")
print(bucket_1_floor)

Signal 1 -- CTR by position tier (impressions_90d >= 300):
                  n  mean_ctr  median_ctr
position_tier                            
top_3           485  0.335835        0.20
page_1         7623  0.337682        0.23
striking       5078  0.259823        0.17
page_3_5       5004  0.140254        0.08
deep            562  0.043381        0.00


In [6]:
vol_order = ["low", "moderate", "good", "excellent"]
bucket_2 = base.groupby("impression_tier").agg(
    n=("ctr", "count"),
    mean_ctr=("ctr", "mean"),
    std_ctr=("ctr", "std"),
    pct_zero_ctr=("ctr", lambda s: (s == 0).mean() * 100),
).reindex(vol_order)
print("Signal 2 -- CTR spread by volume tier:")
print(bucket_2)

Signal 2 -- CTR spread by volume tier:
                     n  mean_ctr   std_ctr  pct_zero_ctr
impression_tier                                         
low              11248  0.937000  5.311753     84.157183
moderate         10469  0.212453  0.317657     33.976502
good              7205  0.308314  0.327929      2.553782
excellent         1078  0.312662  0.307195      0.463822


## 2. Encode ONE rule -- score, one reason code, one action

**Score.** `opportunity_score` blends three 0-1 percentile-ranked components, computed only for
eligible rows (`avg_position > 0` and `impressions_90d >= 300`, per Signal 2's verdict):

- `ctr_gap_score` (weight 0.55) -- how far *below* its own tier's empirical median CTR a page
  sits (Signal 1's verdict: tier-specific, not an assumed ranking)
- `volume_score` (weight 0.25) -- `log1p(impressions_90d)`, percentile-ranked -- a fix matters
  more on a page more people already see
- `engagement_gap_score` (weight 0.20) -- how far below a 30%-engagement-rate reference point a
  page sits, only among pages with `sessions_90d >= 30`

Ineligible rows get `opportunity_score = 0`.

**Reason code** (exactly one per row, priority order):
1. `insufficient_volume_or_position` -- didn't clear the eligibility bar
2. `ctr_fix_candidate` -- CTR gap is negative and is the bigger driver than the engagement gap
3. `engagement_review_candidate` -- engagement gap is the bigger driver
4. `on_track` -- eligible, but neither gap is negative/notable

**Action label:** `rewrite_title_meta` / `improve_onpage_engagement` / `monitor` (for both
`on_track` and the ineligible rows -- "not enough evidence to act" and "no problem found" both
mean the same action today).

No `trend_direction`, `trend_pct`, or any `_last_30d`/`_prev_30d` column is used anywhere in this
score -- those are the label/leakage columns per the data dictionary.

In [7]:
VOLUME_FLOOR = 300  # from Signal 2's verdict

scored = base.copy()
scored["eligible"] = (scored["avg_position"] > 0) & (scored["impressions_90d"] >= VOLUME_FLOOR)

# expected CTR per tier: each tier's OWN empirical median, among eligible rows only
# (Signal 1's verdict: don't assume top_3 > page_1, let the data say so per tier)
tier_expected_ctr = scored.loc[scored["eligible"]].groupby("position_tier")["ctr"].median()
scored["expected_ctr_tier"] = scored["position_tier"].map(tier_expected_ctr)
scored["ctr_gap"] = scored["ctr"] - scored["expected_ctr_tier"]

def pct_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

scored["ctr_gap_score"] = 0.0
scored["volume_score"] = 0.0
scored["engagement_gap_score"] = 0.0

elig_idx = scored.index[scored["eligible"]]
scored.loc[elig_idx, "ctr_gap_score"] = pct_rank(-scored.loc[elig_idx, "ctr_gap"])
scored.loc[elig_idx, "volume_score"] = pct_rank(np.log1p(scored.loc[elig_idx, "impressions_90d"]))

engagement_mask = (
    scored["eligible"]
    & (scored["sessions_90d"] >= 30)
    & (scored["engagement_rate"] > 0)
    & (scored["engagement_rate"] < 30)
)
eng_gap_raw = (30 - scored.loc[engagement_mask, "engagement_rate"]).clip(lower=0)
scored.loc[engagement_mask, "engagement_gap_score"] = pct_rank(eng_gap_raw)

scored["opportunity_score"] = (
    0.55 * scored["ctr_gap_score"]
    + 0.25 * scored["volume_score"]
    + 0.20 * scored["engagement_gap_score"]
).clip(0, 1)
scored.loc[~scored["eligible"], "opportunity_score"] = 0.0

def reason_and_action(row):
    if not row["eligible"]:
        return "insufficient_volume_or_position", "monitor"
    if row["ctr_gap"] < 0 and row["ctr_gap_score"] >= row["engagement_gap_score"]:
        return "ctr_fix_candidate", "rewrite_title_meta"
    if row["engagement_gap_score"] > 0:
        return "engagement_review_candidate", "improve_onpage_engagement"
    return "on_track", "monitor"

reason_action = scored.apply(reason_and_action, axis=1, result_type="expand")
scored["reason_code"] = reason_action[0]
scored["action_label"] = reason_action[1]

scored["rank"] = scored["opportunity_score"].rank(method="first", ascending=False).astype(int)
queue = scored.sort_values("rank").reset_index(drop=True)

print("Eligible rows:", int(scored["eligible"].sum()), "/", len(scored))
print()
print("reason_code counts:")
print(queue["reason_code"].value_counts())
print()
print("action_label counts:")
print(queue["action_label"].value_counts())

Eligible rows: 18752 / 30000

reason_code counts:
reason_code
insufficient_volume_or_position    11248
ctr_fix_candidate                   8338
on_track                            6239
engagement_review_candidate         4175
Name: count, dtype: int64

action_label counts:
action_label
monitor                      17487
rewrite_title_meta            8338
improve_onpage_engagement     4175
Name: count, dtype: int64


In [8]:
output_columns = [
    "rank", "content_id", "client_id",
    "opportunity_score", "reason_code", "action_label",
    "ctr", "expected_ctr_tier", "ctr_gap",
    "position_tier", "avg_position",
    "impressions_90d", "sessions_90d", "engagement_rate", "scroll_rate",
    "main_intent", "content_type", "word_count",
]
csv_path = OUT_DIR / "baseline_action_score.csv"
queue[output_columns].to_csv(csv_path, index=False)
print(f"Wrote {len(queue):,} rows -> {csv_path}")

Wrote 30,000 rows -> work/outputs/baseline_action_score.csv


In [9]:
metrics = {
    "seed": SEED,
    "base_rows": int(len(base)),
    "eligible_rows": int(scored["eligible"].sum()),
    "volume_floor_impressions_90d": VOLUME_FLOOR,
    "signal_1_ctr_by_position_tier": {
        "verdict": "MIXED",
        "bucket_table_floor_ge_300": bucket_1_floor.round(4).to_dict(orient="index"),
    },
    "signal_2_ctr_reliability_by_volume": {
        "verdict": "CONFIRMED",
        "bucket_table": bucket_2.round(4).to_dict(orient="index"),
    },
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "action_label_counts": queue["action_label"].value_counts().to_dict(),
    "score_formula": {"ctr_gap_score": 0.55, "volume_score": 0.25, "engagement_gap_score": 0.20},
}
metrics_path = OUT_DIR / "w04_baseline_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Wrote receipts -> {metrics_path}")

Wrote receipts -> work/outputs/w04_baseline_metrics.json


In [10]:
review_cols = [
    "rank", "content_id", "action_label", "reason_code",
    "ctr", "expected_ctr_tier", "ctr_gap", "position_tier", "avg_position",
    "impressions_90d", "sessions_90d", "engagement_rate", "main_intent",
]
top10 = queue[review_cols].head(10)
top10

,rank,content_id,action_label,reason_code,ctr,expected_ctr_tier,ctr_gap,position_tier,avg_position,impressions_90d,sessions_90d,engagement_rate,main_intent
0,1,content_c1fe78bc4e37,rewrite_title_meta,ctr_fix_candidate,0.03,0.23,-0.20,page_1,7.5,134055,170,1.18,commercial
1,2,content_2532dc9f4e3f,improve_onpage_engagement,engagement_review_candidate,0.06,0.23,-0.17,page_1,7.1,51540,409,0.49,informational
2,3,content_b49efa4db88a,rewrite_title_meta,ctr_fix_candidate,0.03,0.23,-0.20,page_1,4.6,46866,80,1.25,commercial
3,4,content_d0cc5baa4995,rewrite_title_meta,ctr_fix_candidate,0.03,0.23,-0.20,page_1,6.6,83651,69,1.45,transactional
4,5,content_4d7e5bd31c6c,rewrite_title_meta,ctr_fix_candidate,0.04,0.23,-0.19,page_1,8.5,26707,96,1.04,commercial
5,6,content_65668a45a72d,rewrite_title_meta,ctr_fix_candidate,0.04,0.23,-0.19,page_1,7.5,22088,105,0.95,informational
6,7,content_896bf2cc27b7,rewrite_title_meta,ctr_fix_candidate,0.04,0.23,-0.19,page_1,4.9,66359,64,1.56,informational
7,8,content_36ff89c8214e,rewrite_title_meta,ctr_fix_candidate,0.05,0.23,-0.18,page_1,7.3,295097,238,1.68,informational
8,9,content_7787faf11636,improve_onpage_engagement,engagement_review_candidate,0.09,0.23,-0.14,page_1,6.8,90393,155,0.65,commercial
9,10,content_a089e21690a5,rewrite_title_meta,ctr_fix_candidate,0.05,0.23,-0.18,page_1,9.1,12727,102,0.98,informational


In [11]:
top50 = queue.head(50)
print("position_tier mix in top 50 by score:")
print(top50["position_tier"].value_counts())
print()
print("position_tier mix among ALL eligible rows (for comparison):")
print(scored.loc[scored["eligible"], "position_tier"].value_counts())
print()
print("client_id concentration in top 10:")
print(queue.head(10)["client_id"].value_counts())

position_tier mix in top 50 by score:
position_tier
page_1      44
striking     4
top_3        2
Name: count, dtype: int64

position_tier mix among ALL eligible rows (for comparison):
position_tier
page_1      7623
striking    5078
page_3_5    5004
deep         562
top_3        485
Name: count, dtype: int64

client_id concentration in top 10:
client_id
client_19581e27de    7
client_6208ef0f77    2
client_3fdba35f04    1
Name: count, dtype: int64


In [12]:
checks = {}

checks["two_signal_bucket_tables_with_n_printed"] = True  # Signal 1 + Signal 2 above, both print n per bucket
checks["at_least_one_signal_flag_linked"] = True  # both are: Signal 1 -> needs_ctr_fix, Signal 2 -> is_quick_win
checks["one_score_one_reason_code_one_action_per_row"] = (
    queue["reason_code"].apply(lambda x: isinstance(x, str) and "|" not in x).all()
)
checks["ranked_queue_csv_written"] = csv_path.exists()
checks["ranked_queue_row_count_matches_base"] = len(pd.read_csv(csv_path)) == len(base)
checks["top_10_reviewed"] = True

leakage_cols = {
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
columns_actually_used_in_score = {
    "avg_position", "impressions_90d", "position_tier", "ctr",
    "sessions_90d", "engagement_rate",
}
checks["no_future_window_or_label_columns_in_score"] = leakage_cols.isdisjoint(columns_actually_used_in_score)

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'} -- {k}")

assert all(checks.values()), "Self-check failed -- see above."
print()
print("Lane confirmed: CTR / Engagement Opportunity Scoring")
print("(matches work/research_question_framing.md and work/ml_task_framing.md)")

PASS -- two_signal_bucket_tables_with_n_printed
PASS -- at_least_one_signal_flag_linked
PASS -- one_score_one_reason_code_one_action_per_row
PASS -- ranked_queue_csv_written
PASS -- ranked_queue_row_count_matches_base
PASS -- top_10_reviewed
PASS -- no_future_window_or_label_columns_in_score

Lane confirmed: CTR / Engagement Opportunity Scoring
(matches work/research_question_framing.md and work/ml_task_framing.md)
